# 4. Run PyNNLF SA BESS Experiments

This notebook is the manual launch point for the SA BESS PyNNLF experiment. It does not run automatically when opened. Run the cells when you are ready to start or resume the long model batch.

## 1. Setup

Resolve the publication project and the local PyNNLF source tree. The local `src` path is inserted before importing PyNNLF so the notebook uses this repository version, not an older installed package.

In [ ]:
from pathlib import Path
import sys

def find_publication_project(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'scripts' / 'run_sa_bess_batch.py').exists() and (candidate / 'specs' / 'sa_bess_batch.yaml').exists():
            return candidate
    raise FileNotFoundError('Could not find publication/journal_article_1 from the current working directory.')

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'pyproject.toml').exists() and (candidate / 'src' / 'pynnlf').exists():
            return candidate
    raise FileNotFoundError('Could not find the PyNNLF repo root.')

PROJECT_DIR = find_publication_project(Path.cwd().resolve())
REPO_ROOT = find_repo_root(PROJECT_DIR)
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

print(f'Publication project: {PROJECT_DIR}')
print(f'PyNNLF repo root: {REPO_ROOT}')

## 2. Validate Input Datasets

Confirm the three SA BESS datasets are ready for PyNNLF. These checks are quick and do not train any models.

In [ ]:
import pandas as pd

dataset_files = [
    'ds16_sa_bess_underlying_load_30min.csv',
    'ds17_sa_bess_net_load_with_pv_30min.csv',
    'ds18_sa_bess_net_load_with_pv_battery_30min.csv',
]

for filename in dataset_files:
    path = PROJECT_DIR / 'data' / filename
    df = pd.read_csv(path, parse_dates=['datetime'])
    assert list(df.columns) == ['datetime', 'netload_kW'], filename
    assert len(df) == 17_520, filename
    assert df.isna().sum().sum() == 0, filename
    assert df['datetime'].diff().dropna().nunique() == 1, filename
    assert df['datetime'].diff().dropna().iloc[0] == pd.Timedelta(minutes=30), filename
    print(f'{filename}: OK | {df.datetime.min()} to {df.datetime.max()} | rows={len(df):,}')

## 3. Run Or Resume The Full Batch

This is the long-running step. The runner is resumable: completed dataset/model/hyperparameter combinations are skipped on later runs. Plots are disabled by the runner to keep output size down.

In [ ]:
from run_sa_bess_batch import main as run_sa_bess_batch

run_sa_bess_batch()

## 4. Inspect Recap

After the batch finishes, inspect the PyNNLF recap. The complete SA BESS run should have 36 rows: 3 datasets by 12 model/hyperparameter choices.

In [ ]:
recap_path = PROJECT_DIR / 'experiment_result' / 'a1_experiment_result.csv'
recap = pd.read_csv(recap_path)
print(f'Recap rows: {len(recap):,}')
display(recap[['dataset_no', 'model_name', 'test_nRMSE', 'test_nRMSE_stddev']])

## 5. Build Publication Tables

Create the two wide CSV tables for the article: one for test nRMSE and one for test nRMSE standard deviation.

In [ ]:
from process_pynnlf_output import build_publication_tables

nrmse_table, nrmse_stddev_table = build_publication_tables(PROJECT_DIR)
display(nrmse_table)
display(nrmse_stddev_table)
print(PROJECT_DIR / 'results' / '00_data_exploration_and_processing' / 'sa_bess_nrmse_comparison.csv')
print(PROJECT_DIR / 'results' / '00_data_exploration_and_processing' / 'sa_bess_nrmse_stddev_comparison.csv')